# optimizer-init-params-list — ex2: param-groups list-of-dicts with different per-group learning rates

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `optimizer-init-params-list`. Running the final beacon cell reports progress against the `PyTorch: Optimizer init` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Optimizer init` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-init-params-list`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-init-params-list"
DD_SUBTOPIC = "PyTorch: Optimizer init"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Param-groups — list-of-dicts edition

Real PyTorch optimizers don't just take a flat list of params — they take a list of **param-group dicts**, each with its own hyperparameters:

```python
opt = torch.optim.SGD([
    {'params': model.backbone.parameters(), 'lr': 1e-4},
    {'params': model.head.parameters(),     'lr': 1e-2},
], momentum=0.9)
```

Each dict must contain `'params'` (an iterable). Per-group hyperparameters (`lr`, `weight_decay`, etc.) override the optimizer-level default; missing ones fall back to the constructor kwargs. Internally the optimizer materializes each group's `params` into a list (same rule as ex1).

The previous drill (ex1) materialized a single generator into a list. This drill exercises the **per-group-list-of-dicts** layout — building a two-group optimizer that applies different learning rates to different parameter subsets.

### Exercise 2 — param-groups list-of-dicts with different per-group learning rates

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the param-groups pattern (list-of-dicts, each carrying its own `lr`) by building a hand-rolled SGD optimizer whose `.step()` uses the per-group learning rate instead of a single optimizer-wide value.
> Keywords: param-groups, list-of-dicts, per-group-lr, hand-rolled-optimizer
> ```

**KCs targeted:** `optimizer-init-list-vs-generator`, `optimizer-param-groups-list-of-dicts`

Implement `Ex2GroupOptimizer` — a hand-rolled SGD optimizer that accepts a list-of-dicts param-group spec.

**`__init__(self, param_groups)`:**
1. `param_groups` is a list of dicts, each shaped `{'params': iterable, 'lr': float}`.
2. Materialize each group's `'params'` into a list (same rule as ex1, but per-group).
3. Store `self.param_groups` as a list of dicts; each dict must have a `'params'` LIST and an `'lr'` float.

**`@t.no_grad()` `.step(self)`:**
- For each group `g` in `self.param_groups`, iterate `g['params']` and apply `p.data -= g['lr'] * p.grad` for every param with non-None `.grad`.
- Each group uses its OWN `'lr'` — that's the whole point.

**`.zero_grad(self)`:**
- For every param in every group, set `p.grad = None`.

The test sets up two groups (backbone with `lr=0.01`, head with `lr=1.0`) and verifies the head's parameters move 100x further than the backbone's for the same gradient magnitude.

In [ ]:
class Ex2GroupOptimizer:
    """Hand-rolled SGD with per-group learning rates."""

    def __init__(self, param_groups):
        raise NotImplementedError()

    @t.no_grad()
    def step(self):
        raise NotImplementedError()

    def zero_grad(self):
        raise NotImplementedError()


def _test_ex2():
    import torch.nn as nn

    # Build two tiny linear layers — call one 'backbone', one 'head'.
    backbone = nn.Linear(2, 2, bias=False)
    head = nn.Linear(2, 1, bias=False)
    with t.no_grad():
        backbone.weight.fill_(0.0)
        head.weight.fill_(0.0)

    # Build optimizer with two param groups, very different lrs.
    opt = Ex2GroupOptimizer([
        {'params': backbone.parameters(), 'lr': 0.01},
        {'params': head.parameters(),     'lr': 1.0},
    ])

    # Each group's params must be a LIST (materialized).
    assert isinstance(opt.param_groups, list), 'param_groups must be a list'
    assert len(opt.param_groups) == 2, f'expected 2 groups, got {len(opt.param_groups)}'
    for g in opt.param_groups:
        assert isinstance(g, dict)
        assert 'params' in g and 'lr' in g
        assert isinstance(g['params'], list), f'group params must be a list (materialized), got {type(g[chr(39)+"params"+chr(39)])}'

    # Assign identical gradients to backbone and head weights.
    backbone.weight.grad = t.ones_like(backbone.weight)
    head.weight.grad = t.ones_like(head.weight)

    opt.step()

    # Backbone moved by -0.01, head moved by -1.0 — exact per-group lr behavior.
    assert t.allclose(backbone.weight, t.full_like(backbone.weight, -0.01), atol=1e-7), (
        f'backbone should be -0.01, got {backbone.weight}'
    )
    assert t.allclose(head.weight, t.full_like(head.weight, -1.0), atol=1e-7), (
        f'head should be -1.0, got {head.weight}'
    )

    # zero_grad clears every param across every group.
    opt.zero_grad()
    for g in opt.param_groups:
        for p in g['params']:
            assert p.grad is None, f'zero_grad must set grad=None across groups, got {p.grad}'

    # Pass a generator into one group — must still survive multiple .step() calls.
    m = nn.Linear(3, 1, bias=False)
    with t.no_grad(): m.weight.fill_(0.0)
    import types
    g_in = m.parameters()
    assert isinstance(g_in, types.GeneratorType), 'precondition'
    opt2 = Ex2GroupOptimizer([{'params': g_in, 'lr': 0.5}])
    m.weight.grad = t.ones_like(m.weight)
    opt2.step()
    m.weight.grad = t.ones_like(m.weight)
    opt2.step()  # second call MUST also work — group's params must have been materialized.
    assert t.allclose(m.weight, t.full_like(m.weight, -1.0), atol=1e-7), (
        f'after two steps with lr=0.5, weight should be -1.0, got {m.weight}'
    )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
class Ex2GroupOptimizer:
    def __init__(self, param_groups):
        self.param_groups = []
        for g in param_groups:
            self.param_groups.append({
                'params': list(g['params']),
                'lr': g['lr'],
            })

    @t.no_grad()
    def step(self):
        for g in self.param_groups:
            lr = g['lr']
            for p in g['params']:
                if p.grad is not None:
                    p.data -= lr * p.grad

    def zero_grad(self):
        for g in self.param_groups:
            for p in g['params']:
                p.grad = None
```

**Why list-of-dicts instead of a flat list.** Fine-tuning regimes almost always want a smaller learning rate on the pre-trained backbone and a larger one on the freshly initialized head. Param groups make this a 5-line setup instead of two separate optimizers (which would complicate scheduler logic and checkpoint serialization).

**Per-group materialization.** The same generator-exhaustion bug from ex1 applies to EACH group independently — if you write `g['params']` without wrapping in `list(...)`, the first `.step()` consumes the generator and subsequent steps silently skip that group.

**Difference from ex1.** ex1 handled the **single-list** case. ex2 extends to the **list-of-dicts** case with heterogeneous per-group hyperparameters — the canonical fine-tuning pattern.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()